<h1>Содержание<span class="tocSkip"></span></h1>
<div class="toc"><ul class="toc-item"><li><span><a href="#Подготовка" data-toc-modified-id="Подготовка-1"><span class="toc-item-num">1&nbsp;&nbsp;</span>Подготовка</a></span></li><li><span><a href="#Обучение" data-toc-modified-id="Обучение-2"><span class="toc-item-num">2&nbsp;&nbsp;</span>Обучение</a></span></li><li><span><a href="#Выводы" data-toc-modified-id="Выводы-3"><span class="toc-item-num">3&nbsp;&nbsp;</span>Выводы</a></span></li><li><span><a href="#Чек-лист-проверки" data-toc-modified-id="Чек-лист-проверки-4"><span class="toc-item-num">4&nbsp;&nbsp;</span>Чек-лист проверки</a></span></li></ul></div>

# Проект для «Викишоп»

Интернет-магазин «Викишоп» запускает новый сервис. Теперь пользователи могут редактировать и дополнять описания товаров, как в вики-сообществах. То есть клиенты предлагают свои правки и комментируют изменения других. Магазину нужен инструмент, который будет искать токсичные комментарии и отправлять их на модерацию. 

Обучите модель классифицировать комментарии на позитивные и негативные. В вашем распоряжении набор данных с разметкой о токсичности правок.

Постройте модель со значением метрики качества *F1* не меньше 0.75. 

**Инструкция по выполнению проекта**

1. Загрузите и подготовьте данные.
2. Обучите разные модели. 
3. Сделайте выводы.

Для выполнения проекта применять *BERT* необязательно, но вы можете попробовать.

**Описание данных**

Данные находятся в файле `toxic_comments.csv`. Столбец *text* в нём содержит текст комментария, а *toxic* — целевой признак.

## Подготовка

Подгрузим требуемые библиотеки

In [1]:
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords as nltk_stopwords, wordnet
from sklearn.feature_extraction.text import TfidfVectorizer
from nltk.stem import WordNetLemmatizer
from tqdm.notebook import tqdm

from sklearn.model_selection import train_test_split

from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import f1_score

Считаем наш файл и посмотрим правильно ли все считалось

In [2]:
data = pd.read_csv('/datasets/toxic_comments.csv')

In [3]:
display(data.head(10))
data.info()

,Unnamed: 0,text,toxic
0,0,Explanation\nWhy the edits made under my usern...,0
1,1,D'aww! He matches this background colour I'm s...,0
2,2,"Hey man, I'm really not trying to edit war. It...",0
3,3,"""\nMore\nI can't make any real suggestions on ...",0
4,4,"You, sir, are my hero. Any chance you remember...",0
5,5,"""\n\nCongratulations from me as well, use the ...",0
6,6,COCKSUCKER BEFORE YOU PISS AROUND ON MY WORK,1
7,7,Your vandalism to the Matt Shirvington article...,0
8,8,Sorry if the word 'nonsense' was offensive to ...,0
9,9,alignment on this subject and which are contra...,0


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 159292 entries, 0 to 159291
Data columns (total 3 columns):
 #   Column      Non-Null Count   Dtype 
---  ------      --------------   ----- 
 0   Unnamed: 0  159292 non-null  int64 
 1   text        159292 non-null  object
 2   toxic       159292 non-null  int64 
dtypes: int64(2), object(1)
memory usage: 3.6+ MB


In [4]:
print(data.duplicated().sum())

0


Есть один лишний столбец с индексом, его мы удалим, судя по методу инфо пропусков у нас нет, явных дубликатов нет, неявных тоже можно предположить, что нет

In [5]:
data = data.drop('Unnamed: 0',axis=1)

Зададим список стоп-слов, им мы воспользуемся при расчете tf-idf

In [6]:
nltk.download('stopwords')
nltk.download('averaged_perceptron_tagger')
stopwords = set(nltk_stopwords.words('english'))

[nltk_data] Downloading package stopwords to /home/jovyan/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /home/jovyan/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!


In [7]:
def get_wordnet_pos(word):
    tag = nltk.pos_tag([word])[0][1][0].upper()
    tag_dict = {"J": wordnet.ADJ,
                "N": wordnet.NOUN,
                "V": wordnet.VERB, 
                "R": wordnet.ADV} 
    return tag_dict.get(tag, wordnet.NOUN)

Очистим текст от ненужных слов

In [8]:
def clean(text):
    text = text.lower()
    text = re.sub(r'[^a-z!?. ]', '', text)
    text = re.sub(r'\s+', ' ', text)       
    return text.strip()

data['text'] = data['text'].apply(clean)

Лемматизируем текст

In [9]:
lemmatizer = WordNetLemmatizer()
def lemmatize(text):
    res = []
    for word in text.split():
        pos_tag = get_wordnet_pos(word)
        res.append(lemmatizer.lemmatize(word,pos_tag))
    return " ".join(res)

In [13]:
tqdm.pandas()

data['lemm'] = data['text'].progress_apply(lemmatize)

  0%|          | 0/159292 [00:00<?, ?it/s]

In [10]:
# код  ревьюера, проверка качества лемматизации

sentence = "The striped bats are hanging on their feet for best"

lemmatized_correct = "the strip bat be hang on their foot for best"

lemmatized_sentence = lemmatize(sentence)

print("результат твоей лемматизации")
print(lemmatized_sentence)
print('*'*40)
print("должно получиться")
print(lemmatized_correct)

результат твоей лемматизации
The strip bat be hang on their foot for best
****************************************
должно получиться
the strip bat be hang on their foot for best


Разделим наши данные на треннировочную, валидационную и тестовые выборки

In [15]:
df_train_val, df_test = train_test_split(data, test_size=0.20, random_state=42)

df_train, df_val = train_test_split(df_train_val, test_size=0.25, random_state=42)

In [17]:
X_train = df_train['lemm']
X_val = df_val['lemm']
X_test = df_test['lemm']

y_train = df_train['toxic']
y_val = df_val['toxic']
y_test = df_test['toxic']

Преобразуем наши тексты в векторы

In [18]:
count_tf_idf = TfidfVectorizer(stop_words=stopwords) 
tfidf_train = count_tf_idf.fit_transform(X_train)
tfidf_val = count_tf_idf.transform(X_val)
tfidf_test = count_tf_idf.transform(X_test)

## Обучение

Обучим модель логистической регрессии

In [19]:
model1 = LogisticRegression(solver='liblinear',penalty='l1',C=1, random_state=42)
model1.fit(tfidf_train, y_train)

LogisticRegression(C=1, penalty='l1', random_state=42, solver='liblinear')

In [20]:
predictions1 = model1.predict(tfidf_val)

In [21]:
print('F1-мера для логистической регрессии: ',f1_score(y_val,predictions1))

F1-мера для логистической регрессии:  0.7655435915248351


Метрика удовлетворяет нашим требованиям, так что можно считать, что логистическая регрессия справилась

Обучим еще модель

In [22]:
model2 = SVC(kernel='linear', random_state=42)
model2.fit(tfidf_train, y_train)

predictions2 = model2.predict(tfidf_val)

print('F1-мера для SVC: ',f1_score(y_val,predictions2))

F1-мера для SVC:  0.7654666200629151


In [24]:
final_predictions = model1.predict(tfidf_test)
print('F1-мера для SVC: ',f1_score(y_test,final_predictions))

F1-мера для SVC:  0.7560014331780723


## Выводы

Модели логистической регрессии хватило, даже не потребовалось создавать дополнительные признаки, метрика получилась равной 0.7655435915248351. Аналогично логистической справилась и SVC 0.7654666200629151. Поэтому мы выбрали логистическую регрессию. По итогу метрика получилась 0.7560014331780723. В данном проекте мы очистили текст от лишних слов, лемматизировали его и преобразовали в вектор с учетом стоп-слов

## Чек-лист проверки

- [x]  Jupyter Notebook открыт
- [x]  Весь код выполняется без ошибок
- [x]  Ячейки с кодом расположены в порядке исполнения
- [x]  Данные загружены и подготовлены
- [x]  Модели обучены
- [x]  Значение метрики *F1* не меньше 0.75
- [x]  Выводы написаны